<a href="https://colab.research.google.com/github/Almas1989/PySpark_colab_practice/blob/main/Getting_Started_with_PySpark.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Начало работы с PySpark в Google Colab

PySpark - это Python интерфейс для Apache Spark. Основные сценарии использования PySpark - работа с огромными объемами данных и создание конвейеров обработки данных.

Вам не нужно работать с большими данными, чтобы получить выгоду от PySpark. Я считаю, что SparkSQL - отличный инструмент для выполнения рутинного анализа данных. Pandas может работать медленно, и вы можете обнаружить, что пишете много кода для очистки данных, тогда как те же действия занимают гораздо меньше кода в SQL. Давайте начнем!

Подробнее здесь! http://spark.apache.org/docs/latest/api/python/

# 1. Установка PySpark в Google Colab

In [1]:
!sudo apt update
!apt-get install openjdk-8-jdk-headless -qq > /dev/null
# Проверьте этот сайт для последней ссылки на загрузку https://www.apache.org/dyn/closer.lua/spark/spark-3.2.1/spark-3.2.1-bin-hadoop3.2.tgz
!wget -q https://dlcdn.apache.org/spark/spark-3.2.1/spark-3.2.1-bin-hadoop3.2.tgz
!tar xf spark-3.2.1-bin-hadoop3.2.tgz
!pip install -q findspark
!pip install pyspark
!pip install py4j

import os
import sys
# os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
# os.environ["SPARK_HOME"] = "/content/spark-3.2.1-bin-hadoop3.2"


import findspark
findspark.init()
findspark.find()

import pyspark

from pyspark.sql import DataFrame, SparkSession
from typing import List
import pyspark.sql.types as T
import pyspark.sql.functions as F

spark= SparkSession \
       .builder \
       .appName("Наш первый Spark пример") \
       .getOrCreate()

spark

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]      
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease                         
Get:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]           
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]        
Hit:7 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease   
Hit:8 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease    
Get:9 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:10 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,860 kB]
Get:11 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [9,573 kB]
Get:12 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [3,633 kB]
Get:13 http://archive.ubuntu.com/ubuntu jammy-upd

In [2]:
spark

# 2. Чтение данных

Для этого примера я буду использовать публично доступный набор данных в формате CSV.

In [3]:
import requests
path = "https://raw.githubusercontent.com/owid/covid-19-data/master/public/data/owid-covid-data.csv"
req = requests.get(path)
url_content = req.content

csv_file_name = 'owid-covid-data.csv'
csv_file = open(csv_file_name, 'wb')

csv_file.write(url_content)
csv_file.close()

df = spark.read.csv('/content/'+csv_file_name, header=True, inferSchema=True)

# 3. PySpark DataFrames

In [4]:
# Просмотр схемы dataframe
df.printSchema()

root
 |-- iso_code: string (nullable = true)
 |-- continent: string (nullable = true)
 |-- location: string (nullable = true)
 |-- date: date (nullable = true)
 |-- total_cases: integer (nullable = true)
 |-- new_cases: integer (nullable = true)
 |-- new_cases_smoothed: double (nullable = true)
 |-- total_deaths: integer (nullable = true)
 |-- new_deaths: integer (nullable = true)
 |-- new_deaths_smoothed: double (nullable = true)
 |-- total_cases_per_million: double (nullable = true)
 |-- new_cases_per_million: double (nullable = true)
 |-- new_cases_smoothed_per_million: double (nullable = true)
 |-- total_deaths_per_million: double (nullable = true)
 |-- new_deaths_per_million: double (nullable = true)
 |-- new_deaths_smoothed_per_million: double (nullable = true)
 |-- reproduction_rate: double (nullable = true)
 |-- icu_patients: integer (nullable = true)
 |-- icu_patients_per_million: double (nullable = true)
 |-- hosp_patients: integer (nullable = true)
 |-- hosp_patients_per_mil

In [5]:
# Преобразование колонки с датой
df.select(F.to_date(df.date).alias('date'))

DataFrame[date: date]

In [6]:
# Сводная статистика
df.describe().show()

+-------+--------+-------------+-----------+--------------------+------------------+------------------+------------------+------------------+-------------------+-----------------------+---------------------+------------------------------+------------------------+----------------------+-------------------------------+------------------+------------------+------------------------+------------------+-------------------------+---------------------+---------------------------------+----------------------+----------------------------------+-------------------+------------------+------------------------+----------------------+------------------+-------------------------------+-------------------+------------------+-------------+--------------------+--------------------+-----------------------+--------------------+------------------+-------------------------+------------------------------+-----------------------------+-----------------------------------+--------------------------+-------------

In [ ]:
# Фильтрация DataFrame
df.filter(df.location == "United States").orderBy(F.desc("date")).show()

In [ ]:
# Простая функция Group By
df.groupBy("location").sum("new_cases").orderBy(F.desc("sum(new_cases)")).show(truncate=False)

# 4. Spark SQL

Что мне действительно нравится в модуле SQL, так это то, что с его помощью очень легко взаимодействовать с данными, продолжая использовать Spark. Нужно меньше изучать, поскольку это в основном тот же синтаксис SQL, с которым вы уже могли быть знакомы.

In [ ]:
# Создание таблицы из dataframe
df.createOrReplaceTempView("covid_data")  # временное представление
# df.saveAsTable("covid_data")  # Сохранить как таблицу
# df.write.mode("overwrite").saveAsTable("covid_data")  # Сохранить как таблицу и перезаписать, если существует

In [10]:

df2 = spark.sql("SELECT * from covid_data")
df2.printSchema()
df2.show()

root
 |-- iso_code: string (nullable = true)
 |-- continent: string (nullable = true)
 |-- location: string (nullable = true)
 |-- date: date (nullable = true)
 |-- total_cases: integer (nullable = true)
 |-- new_cases: integer (nullable = true)
 |-- new_cases_smoothed: double (nullable = true)
 |-- total_deaths: integer (nullable = true)
 |-- new_deaths: integer (nullable = true)
 |-- new_deaths_smoothed: double (nullable = true)
 |-- total_cases_per_million: double (nullable = true)
 |-- new_cases_per_million: double (nullable = true)
 |-- new_cases_smoothed_per_million: double (nullable = true)
 |-- total_deaths_per_million: double (nullable = true)
 |-- new_deaths_per_million: double (nullable = true)
 |-- new_deaths_smoothed_per_million: double (nullable = true)
 |-- reproduction_rate: double (nullable = true)
 |-- icu_patients: integer (nullable = true)
 |-- icu_patients_per_million: double (nullable = true)
 |-- hosp_patients: integer (nullable = true)
 |-- hosp_patients_per_mil

In [11]:
groupDF = spark.sql("SELECT location, count(*) from covid_data group by location")
groupDF.show()

+--------------------+--------+
|            location|count(1)|
+--------------------+--------+
|                Chad|    1674|
|            Anguilla|    1674|
|            Kiribati|    1674|
|              Guyana|    1674|
|             Eritrea|    1674|
|              Jersey|    1674|
|            Djibouti|    1674|
|                Fiji|    1674|
|                Iraq|    1674|
|              Europe|    1684|
|             Germany|    1674|
|             Comoros|    1674|
|         Afghanistan|    1674|
|            Cambodia|    1674|
|High-income count...|    3026|
|              Jordan|    1674|
|              France|    1674|
|              Greece|    1674|
|              Kosovo|    1674|
|              Africa|    1674|
+--------------------+--------+
only showing top 20 rows


# 5. Пример с другим набором данных
Этот набор данных поставляется с вашей сессией Google Colab

In [12]:
df = spark.read.csv("/content/sample_data/california_housing_train.csv", header=True, inferSchema=True)

In [13]:
df.printSchema()

root
 |-- longitude: double (nullable = true)
 |-- latitude: double (nullable = true)
 |-- housing_median_age: double (nullable = true)
 |-- total_rooms: double (nullable = true)
 |-- total_bedrooms: double (nullable = true)
 |-- population: double (nullable = true)
 |-- households: double (nullable = true)
 |-- median_income: double (nullable = true)
 |-- median_house_value: double (nullable = true)



In [ ]:
# Вывести N строк
df.show(5)

In [15]:
df.count()

17000

In [16]:
df.select("housing_median_age","total_rooms").show(5)

+------------------+-----------+
|housing_median_age|total_rooms|
+------------------+-----------+
|              15.0|     5612.0|
|              19.0|     7650.0|
|              17.0|      720.0|
|              14.0|     1501.0|
|              20.0|     1454.0|
+------------------+-----------+
only showing top 5 rows


In [17]:
df.describe().show()

+-------+-------------------+------------------+------------------+-----------------+-----------------+------------------+-----------------+------------------+------------------+
|summary|          longitude|          latitude|housing_median_age|      total_rooms|   total_bedrooms|        population|       households|     median_income|median_house_value|
+-------+-------------------+------------------+------------------+-----------------+-----------------+------------------+-----------------+------------------+------------------+
|  count|              17000|             17000|             17000|            17000|            17000|             17000|            17000|             17000|             17000|
|   mean|-119.56210823529375|  35.6252247058827| 28.58935294117647|2643.664411764706|539.4108235294118|1429.5739411764705|501.2219411764706| 3.883578100000021|207300.91235294117|
| stddev| 2.0051664084260357|2.1373397946570867|12.586936981660406|2179.947071452777|421.4994515798648| 1

In [18]:
df.select('total_rooms').distinct().show()

+-----------+
|total_rooms|
+-----------+
|      934.0|
|     3980.0|
|     4142.0|
|      596.0|
|     1761.0|
|     5983.0|
|     2815.0|
|     6433.0|
|      299.0|
|     2734.0|
|      769.0|
|     1051.0|
|     7554.0|
|     4066.0|
|     2862.0|
|     3597.0|
|      692.0|
|      720.0|
|     1765.0|
|     2523.0|
+-----------+
only showing top 20 rows


In [19]:
from pyspark.sql import functions as F
test = df.groupBy('total_rooms').agg(F.sum('housing_median_age'))

In [20]:
test.toPandas()

,total_rooms,sum(housing_median_age)
0,934.0,135.0
1,3980.0,25.0
2,4142.0,37.0
3,596.0,25.0
4,1761.0,154.0
...,...,...
5528,3620.0,18.0
5529,947.0,62.0
5530,710.0,52.0
5531,91.0,43.0


In [ ]:
# Подсчет и удаление пропущенных значений

df.select([F.count(F.when(F.isnull(c), c)).alias(c) for c in df.columns]).show()

# Создание тестового Spark DataFrame

In [22]:
data = [
        ('John','Smith',1),
        ('Jane','Smith',2),
        ('Jonas','Smith',3),
]

columns = ["firstname","middlename","lastname","dob","gender","salary"]
df = spark.createDataFrame(data=data, schema = columns)

IndexError: list index out of range

In [ ]:
df

# Советы и хитрости Spark

Это коллекция фрагментов кода для распространенных или сложных задач

## Pandas DataFrame в Spark DataFrame

In [ ]:
import pandas as pd
import numpy as np

df = pd.DataFrame(np.random.randint(100,size=(1000, 3)),columns=['A','B','C'])
spark_df = spark.createDataFrame(df)
spark_df.show()

In [ ]:
# Преобразование объектных колонок в pandas dataframe в строки
for i in df.select_dtypes(include='object').columns.tolist():
	df[i] = df[i].astype(str)

# Преобразование datetime в UTC
  for i in [col for col in df.columns if df[col].dtype == 'datetime64[ns]']:
   df[i] = pd.to_datetime(df[i], utc=True)

# Замена nan и "None" в pandas dataframe на null в spark dataframe
spark_df = spark.createDataFrame(df).replace('None', None).replace(float('nan'), None)

## Оконные функции

In [ ]:
data = [
        (1,'2021-01-01 10:00:00'),
        (1,'2021-01-01 11:00:00'),
        (1,'2021-01-01 12:00:00'),
        (2,'2021-01-01 12:00:00'),
        (2,'2021-01-01 13:00:00'),
        (2,'2021-01-01 14:00:00'),
]

columns = ["id","datetime"]
df = spark.createDataFrame(data=data, schema = columns)
df.createOrReplaceTempView("window_test")
df.show()

In [ ]:
# Выбор минимума и максимума по определенной группе
spark.sql('''
Select
  id,

  max(datetime) OVER (Partition BY id ORDER BY datetime) as max_date,
  min(datetime) OVER (Partition BY id ORDER BY datetime) as min_date,

  ROW_NUMBER() OVER (Partition BY id ORDER BY datetime) as row_number

  FROM window_test

''').show()

In [ ]:
# Выбор номера строки или порядкового ранга для каждой строки в указанной группе.
# Это отлично подходит для подранжирования в таблице

spark.sql('''
Select
  id,
  datetime,

  ROW_NUMBER() OVER (Partition BY id ORDER BY datetime) as row_number

  FROM window_test

''').show()

## Дедупликация данных путем возврата наиболее недавно обновленной строки с использованием оконной функции

In [ ]:
data = [
        (1,'2021-01-01',100,'A'),
        (1,'2021-01-31',105,'A'),
        (2,'2021-02-04',160,'B'),
        (2,'2021-02-07',145,'B'),
]

columns = ["id","date","score","type"]
df = spark.createDataFrame(data=data, schema = columns)
df.createOrReplaceTempView("window_test")
df.show()

In [ ]:
df2 = spark.sql("""
WITH T AS (
  SELECT
  *,
  ROW_NUMBER() OVER (PARTITION BY id ORDER BY date DESC) AS version_number
  FROM window_test
)

SELECT * FROM T WHERE version_number = 1;

""")

df2.show()

In [ ]:
spark.sql("""
  SELECT
  *,
  SUM(score) OVER (PARTITION by type ORDER BY date) as score_cumulative
  FROM window_test

""").show()

## Ограничение количества результатов на группу с помощью оконной функции

In [ ]:
import pandas as pd
import numpy as np

df = pd.DataFrame(
np.hstack((
    np.random.randint(1,5,size=(100000, 1)),
    np.random.randint(100,size=(100000, 1))
))
, columns=['company_id', 'number'])

dff = spark.createDataFrame(df)
dff.createOrReplaceTempView("window_test_limits")


In [ ]:
spark.sql("""
WITH T AS (
  SELECT
    company_id,
    number,
    ROW_NUMBER() OVER (PARTITION BY company_id ORDER BY number) AS row_number
  FROM window_test_limits
    )

SELECT * FROM T WHERE row_number <= 100

""").show()

## Расчет 7-дневной скользящей средней

In [ ]:
df = pd.DataFrame(pd.date_range('1/1/2022','1/31/2022',freq='D'), columns=['date'])
import random
df['company_id'] = 1
df['number'] = df.apply(lambda x: random.randint(0,100), axis = 1)

dff = spark.createDataFrame(df)
dff.createOrReplaceTempView("window_data")

dff.show()

In [ ]:
spark.sql("""
SELECT
  date,
  company_id,
  number,
  AVG(number) OVER (PARTITION BY company_id ORDER BY date ASC RANGE BETWEEN INTERVAL 6 DAYS PRECEDING AND CURRENT ROW) as last_7_day_avg
FROM window_data
""").show()

## Месячные активные пользователи

In [ ]:
import pandas as pd
df = pd.DataFrame(pd.date_range('1/1/2022','1/31/2022',freq='D'), columns=['login_date'])
import random
df['company_id'] = 1
df['user_id'] = df.apply(lambda x: random.randint(0,3), axis = 1)

dff = spark.createDataFrame(df)
dff.createOrReplaceTempView("users_data")

dff.show()

In [ ]:
# Пересмотреть эту трансформацию
spark.sql("""
SELECT
  login_date,
  COUNT(user_id) OVER (PARTITION BY login_date ORDER BY login_date ASC RANGE BETWEEN INTERVAL 30 DAYS PRECEDING AND CURRENT ROW) AS monthly_active_users
  FROM users_data
""").show()

## Поиск разницы во времени между связанными строками с использованием оконной функции

In [ ]:
data = [
        (1,'start','2021-01-01',100,'A'),
        (1,'end','2021-01-31',200,'A'),
        (2,'start','2021-03-05 4:53:11',100,'A'),
        (2,'end','2021-05-01 05:06:38',200,'A'),
]

columns = ["id","session","datetime","station_return","type"]
df = spark.createDataFrame(data=data, schema = columns)
df.createOrReplaceTempView("window_test")
df.show()

In [ ]:
spark.sql('''
SELECT
  id,
  datetime,
  lead(datetime) OVER (PARTITION BY id ORDER BY datetime) as next_datetime,
  DATEDIFF(lead(datetime) OVER (PARTITION BY id ORDER BY datetime),datetime) as duration_in_days

FROM window_test

''').show()

## Разворот (Unpivot)

In [ ]:
from pyspark.sql.types import *


data = [
        ('tim', 10, 9, 8, 5),
        ('john', 5, 6, 3, 6),
        ('jane', 7, 8, 9, 10),

]

schema = StructType([
   StructField("name", StringType(), True),
   StructField("experience", IntegerType(), True),
   StructField("satisfaction", IntegerType(), True),
   StructField("customer_service", IntegerType(), True),
   StructField("speed_of_service", IntegerType(), True)])


df = spark.createDataFrame(data, schema=schema)

df.show()

In [ ]:
cols = ['experience', 'satisfaction', 'customer_service', 'speed_of_service']

exprs = f"""stack({len(cols)}, {", ".join([f"'{i}',{i}" for i in cols])}) as (question,score)"""

unpivotted_df = df.select("name",F.expr(exprs))

unpivotted_df.show()

## Замена значений с использованием словаря

In [ ]:
df = (spark
    .createDataFrame([
        (1, 'hello',3),
        (2, 'hello',5),
        (3, 'hello',5),
        (135246, 'hello',4),
        (54936, 'hello',4)
        ],
        ["id", "text","num"]))

In [ ]:
mapping = {
1: 5555,
4:9999
}

In [ ]:
df.replace(mapping,1,'id').replace(mapping,1,'_3').show()

## Создание диапазона дат

In [ ]:
date_range_df = spark.sql("SELECT explode(sequence(to_date('2018-01-01'), to_date('2018-03-01'), interval 1 day)) as date")
date_range_df.show()

## Объединение значений строк после группировки

In [ ]:
df = (spark
    .createDataFrame([
        (1, 'hello',3),
        (2, 'hello',5),
        (3, 'hello',5),
        (3, 'hello',5),
        (3, 'hello',5),
        ],
        ["id", "text"]))

df.createOrReplaceTempView("group_array")

df.show()

In [ ]:
# Вернуть каждый элемент
spark.sql("Select g.text, collect_list(g.id) FROM group_array as g GROUP BY 1").show()

In [ ]:
# Вернуть уникальный список
spark.sql("Select g.text, collect_set(g.id) FROM group_array as g GROUP BY 1").show()

## Переименование колонок Spark с помощью словаря

In [ ]:
col_dict = {
    'id':'ID',
    'test':'hello'
}

# Выбрать только определенные колонки из файла
df = spark.read.parquet(path).select([k for k in cols_2016])

# Переименовать колонки
for old_name, new_name in col_dict.items():
  df = df.withColumnRenamed(old_name,new_name)

df.createOrReplaceTempView("test")

test.show()

## Чтение нескольких Parquet файлов в один Spark DataFrame

In [ ]:
import glob

parquet_files = glob.glob('/content/*.parquet')
# * это подстановочный символ

df = spark.read.parquet(*parquet_files)

## Разделение и получение последнего элемента в Spark SQL

In [ ]:
spark.sql("""
SELECT
  "This.is.a.test" AS text,
  SPLIT("This.is.a.test",'[\.]') AS split,
  REVERSE(SPLIT("This.is.a.test",'[\.]'))[0] AS last_word
""").show()

## Обработка NULL значений

In [ ]:
df = (spark
    .createDataFrame([
        (1, 'hello',None),
        (2, 'hello',None),
        (3, 'hello',5),
        (3, 'hello',5),
        (3, 'hello',5),
        ],
        ["id", "text"]))

df.createOrReplaceTempView("group_array")

df.show()

In [ ]:
spark.sql("Select * from group_array where _3 IS NOT NULL").show()

## Использование JDBC драйвера

In [ ]:
!pip install JayDeBeApi
import jaydebeapi
import os

# Загрузка JDBC драйверов
!wget https://repo1.maven.org/maven2/org/apache/hive/hive-jdbc/2.3.7/hive-jdbc-2.3.7-standalone.jar
!zip -q -d hive-jdbc-2.3.7-standalone.jar org/apache/logging/log4j/core/lookup/JndiLookup.class
!unzip hive-jdbc-2.3.7-standalone.jar > output.txt

!wget https://github.com/timveil/hive-jdbc-uber-jar/releases/download/v1.8-2.6.3/hive-jdbc-uber-2.6.3.0-235.jar


DRIVER_CLASS = 'org.apache.hive.jdbc.HiveDriver'
DRIVER_PATH = 'hive-jdbc-2.3.7-standalone.jar'
ASCEND_ENV = 'trial'
CONN_URL = 'jdbc:'

user = 'admin'
pw = 'admin'

conn = jdbc.connect(DRIVER_CLASS,
                    CONN_URL,
                    [user, pw],
                    DRIVER_PATH)

# Регулярные выражения (Regex)

In [ ]:
spark.sql("""
SELECT
  '(5) Strongly Agree',
  regexp_extract('(10) Strongly Agree', '([0-9]+)')
""").show()

# User Defined Functions (UDF)

UDF позволяет создавать пользовательские функции для применения к данным в DataFrame

In [ ]:
# Пример 1: Простая UDF для преобразования текста в верхний регистр
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType, IntegerType

# Определяем функцию Python
def to_upper(text):
    return text.upper() if text else None

# Регистрируем как UDF
upper_udf = udf(to_upper, StringType())

# Создаем тестовый DataFrame
test_data = [
    (1, 'hello'),
    (2, 'world'),
    (3, 'pyspark')
]
df_test = spark.createDataFrame(test_data, ["id", "text"])

# Применяем UDF
df_test.withColumn("text_upper", upper_udf(df_test.text)).show()

In [ ]:
# Пример 2: UDF с декоратором и регистрация для SQL
from pyspark.sql.functions import udf
from pyspark.sql.types import IntegerType

@udf(returnType=IntegerType())
def calculate_age_category(age):
    """Категоризация возраста"""
    if age is None:
        return None
    elif age < 18:
        return 1  # Молодой
    elif age < 60:
        return 2  # Взрослый
    else:
        return 3  # Пожилой

# Регистрируем UDF для использования в SQL
spark.udf.register("age_category_sql", calculate_age_category)

# Тестовые данные
age_data = [(1, 15), (2, 35), (3, 65), (4, 28)]
df_ages = spark.createDataFrame(age_data, ["id", "age"])

# Использование через DataFrame API
df_ages.withColumn("category", calculate_age_category(df_ages.age)).show()

# Использование через SQL
df_ages.createOrReplaceTempView("ages_table")
spark.sql("SELECT id, age, age_category_sql(age) as category FROM ages_table").show()

# Joins - Объединение DataFrame

Joins позволяют объединять несколько DataFrame по общим ключам. PySpark поддерживает различные типы объединений

In [ ]:
# Создаем тестовые DataFrame для демонстрации Joins
# DataFrame с пользователями
users_data = [
    (1, 'Alice', 'IT'),
    (2, 'Bob', 'HR'),
    (3, 'Charlie', 'IT'),
    (4, 'Diana', 'Finance')
]
df_users = spark.createDataFrame(users_data, ["user_id", "name", "department"])

# DataFrame с зарплатами
salaries_data = [
    (1, 70000),
    (2, 60000),
    (3, 75000),
    (5, 80000)  # user_id 5 не существует в users
]
df_salaries = spark.createDataFrame(salaries_data, ["user_id", "salary"])

print("Users DataFrame:")
df_users.show()

print("Salaries DataFrame:")
df_salaries.show()

In [ ]:
# 1. INNER JOIN - возвращает только совпадающие записи из обоих DataFrame
print("INNER JOIN:")
df_users.join(df_salaries, on="user_id", how="inner").show()

# 2. LEFT JOIN (LEFT OUTER) - все записи из левого DF + совпадения из правого
print("\nLEFT JOIN:")
df_users.join(df_salaries, on="user_id", how="left").show()

# 3. RIGHT JOIN (RIGHT OUTER) - все записи из правого DF + совпадения из левого
print("\nRIGHT JOIN:")
df_users.join(df_salaries, on="user_id", how="right").show()

# 4. FULL OUTER JOIN - все записи из обоих DataFrame
print("\nFULL OUTER JOIN:")
df_users.join(df_salaries, on="user_id", how="outer").show()

In [ ]:
# Joins с использованием SQL
df_users.createOrReplaceTempView("users")
df_salaries.createOrReplaceTempView("salaries")

print("INNER JOIN через SQL:")
spark.sql("""
    SELECT u.user_id, u.name, u.department, s.salary
    FROM users u
    INNER JOIN salaries s ON u.user_id = s.user_id
""").show()

print("\nLEFT JOIN через SQL:")
spark.sql("""
    SELECT u.user_id, u.name, u.department, s.salary
    FROM users u
    LEFT JOIN salaries s ON u.user_id = s.user_id
""").show()

# Работа с JSON данными

PySpark предоставляет мощные инструменты для работы с JSON: чтение файлов, парсинг JSON-строк и распаковка вложенных структур

In [ ]:
# Пример 1: Создание DataFrame с JSON-строками
from pyspark.sql.functions import from_json, to_json, col, explode
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, ArrayType

# Тестовые данные с JSON-строками
json_data = [
    (1, '{"name": "John", "age": 30, "city": "New York"}'),
    (2, '{"name": "Jane", "age": 25, "city": "Paris"}'),
    (3, '{"name": "Bob", "age": 35, "city": "London"}')
]
df_json_strings = spark.createDataFrame(json_data, ["id", "json_str"])

print("DataFrame с JSON-строками:")
df_json_strings.show(truncate=False)

In [ ]:
# Пример 2: Парсинг JSON-строк в структурированные колонки
# Определяем схему JSON
json_schema = StructType([
    StructField("name", StringType(), True),
    StructField("age", IntegerType(), True),
    StructField("city", StringType(), True)
])

# Распаковываем JSON в отдельные колонки
df_parsed = df_json_strings.withColumn("parsed_data", from_json(col("json_str"), json_schema))

print("Распакованный JSON:")
df_parsed.select("id", "parsed_data.*").show()

# Преобразование обратно в JSON
df_to_json = df_parsed.select("id", to_json(col("parsed_data")).alias("json_output"))
print("\nОбратное преобразование в JSON:")
df_to_json.show(truncate=False)

In [ ]:
# Пример 3: Работа с вложенными JSON и массивами
nested_json_data = [
    (1, '{"name": "Alice", "skills": ["Python", "Spark", "SQL"]}'),
    (2, '{"name": "Bob", "skills": ["Java", "Kafka"]}'),
    (3, '{"name": "Charlie", "skills": ["Python", "Machine Learning", "R"]}')
]
df_nested = spark.createDataFrame(nested_json_data, ["id", "json_str"])

# Схема с массивом
nested_schema = StructType([
    StructField("name", StringType(), True),
    StructField("skills", ArrayType(StringType()), True)
])

# Парсинг и распаковка массива
df_nested_parsed = df_nested.withColumn("data", from_json(col("json_str"), nested_schema))

print("Вложенный JSON с массивом:")
df_nested_parsed.select("id", "data.*").show(truncate=False)

# Используем explode для развертывания массива
print("\nРазвернутый массив навыков:")
df_nested_parsed.select("id", "data.name", explode("data.skills").alias("skill")).show()

# Caching и Persistence

Кэширование позволяет сохранять промежуточные результаты в памяти или на диске для повторного использования, что значительно ускоряет вычисления

In [ ]:
# Создаем большой DataFrame для демонстрации кэширования
import time

large_data = [(i, f"name_{i}", i * 100) for i in range(1, 10001)]
df_large = spark.createDataFrame(large_data, ["id", "name", "value"])

print("DataFrame создан")

# Без кэширования - каждый раз происходит пересчет
start_time = time.time()
count1 = df_large.filter(col("value") > 50000).count()
time1 = time.time() - start_time

start_time = time.time()
count2 = df_large.filter(col("value") > 50000).count()
time2 = time.time() - start_time

print(f"\nБез кэша:")
print(f"Первый запрос: {time1:.4f} сек, результат: {count1}")
print(f"Второй запрос: {time2:.4f} сек, результат: {count2}")

In [ ]:
# Теперь с кэшированием
df_cached = df_large.cache()  # или df_large.persist()

# Первый запрос - данные кэшируются
start_time = time.time()
count3 = df_cached.filter(col("value") > 50000).count()
time3 = time.time() - start_time

# Второй запрос - данные берутся из кэша (быстрее!)
start_time = time.time()
count4 = df_cached.filter(col("value") > 50000).count()
time4 = time.time() - start_time

print(f"\nС кэшем:")
print(f"Первый запрос (кэширование): {time3:.4f} сек, результат: {count3}")
print(f"Второй запрос (из кэша): {time4:.4f} сек, результат: {count4}")
print(f"Ускорение: {time3/time4:.2f}x")

# Очистка кэша
df_cached.unpersist()
print("\nКэш очищен")

In [ ]:
# Различные уровни persistence
from pyspark import StorageLevel

# MEMORY_ONLY - только в памяти (по умолчанию для cache())
df_memory_only = df_large.persist(StorageLevel.MEMORY_ONLY)

# MEMORY_AND_DISK - сначала память, потом диск
df_memory_disk = df_large.persist(StorageLevel.MEMORY_AND_DISK)

# DISK_ONLY - только на диске
df_disk_only = df_large.persist(StorageLevel.DISK_ONLY)

print("Доступные уровни persistence:")
print("- MEMORY_ONLY: данные только в памяти")
print("- MEMORY_AND_DISK: данные в памяти и на диске")  
print("- DISK_ONLY: данные только на диске")
print("- MEMORY_ONLY_SER: сериализованные данные в памяти")
print("- MEMORY_AND_DISK_SER: сериализованные данные в памяти и на диске")

# Очистка
df_memory_only.unpersist()
df_memory_disk.unpersist()
df_disk_only.unpersist()

# Partitioning - Разделение данных

Partitioning позволяет распределить данные по нескольким разделам для параллельной обработки и оптимизации производительности

In [ ]:
# Создаем DataFrame для демонстрации partitioning
partition_data = [(i, f"category_{i % 5}", i * 10) for i in range(1, 101)]
df_part = spark.createDataFrame(partition_data, ["id", "category", "value"])

# Проверяем текущее количество разделов
print(f"Текущее количество разделов: {df_part.rdd.getNumPartitions()}")

# Repartition - увеличиваем количество разделов (полная перетасовка данных)
df_repartitioned = df_part.repartition(10)
print(f"После repartition(10): {df_repartitioned.rdd.getNumPartitions()} разделов")

# Coalesce - уменьшаем количество разделов (без полной перетасовки, эффективнее)
df_coalesced = df_repartitioned.coalesce(5)
print(f"После coalesce(5): {df_coalesced.rdd.getNumPartitions()} разделов")

In [ ]:
# Partitioning по колонке - данные с одинаковым значением попадут в один раздел
df_partitioned_by_col = df_part.repartition(5, "category")
print(f"\nРазделение по колонке 'category': {df_partitioned_by_col.rdd.getNumPartitions()} разделов")

# Это полезно для оптимизации группировок и joins
# Данные одной категории будут в одном разделе
print("\nПример данных после разделения по category:")
df_partitioned_by_col.show(10)

# Можно посмотреть распределение данных по разделам
def count_partition(iterator):
    yield sum(1 for _ in iterator)

partition_counts = df_partitioned_by_col.rdd.mapPartitions(count_partition).collect()
print(f"\nРаспределение записей по разделам: {partition_counts}")

# Broadcast Variables

Broadcast переменные позволяют эффективно распространять небольшие данные (справочники, lookup таблицы) на все узлы кластера для оптимизации joins

In [ ]:
# Пример: большая таблица транзакций и маленький справочник категорий
from pyspark.sql.functions import broadcast

# Большая таблица транзакций
transactions_data = [(i, i % 10, i * 100) for i in range(1, 10001)]
df_transactions = spark.createDataFrame(transactions_data, ["transaction_id", "category_id", "amount"])

# Маленькая таблица-справочник категорий
categories_data = [
    (1, 'Electronics'),
    (2, 'Clothing'),
    (3, 'Food'),
    (4, 'Books'),
    (5, 'Toys'),
    (6, 'Sports'),
    (7, 'Home'),
    (8, 'Beauty'),
    (9, 'Automotive')
]
df_categories = spark.createDataFrame(categories_data, ["category_id", "category_name"])

print("Большая таблица транзакций:")
df_transactions.show(5)
print(f"Количество записей: {df_transactions.count()}")

print("\nМаленькая таблица категорий:")
df_categories.show()
print(f"Количество записей: {df_categories.count()}")

In [ ]:
# Обычный join (без broadcast)
print("Обычный JOIN:")
df_regular_join = df_transactions.join(df_categories, on="category_id", how="inner")
df_regular_join.show(10)

# Broadcast join - оптимизированный для маленьких таблиц
print("\nBROADCAST JOIN (оптимизированный):")
df_broadcast_join = df_transactions.join(broadcast(df_categories), on="category_id", how="inner")
df_broadcast_join.show(10)

# Broadcast join работает быстрее, потому что:
# 1. Маленькая таблица копируется на все узлы кластера один раз
# 2. Не требуется shuffle больших данных
# 3. Join происходит локально на каждом узле

print("\n✓ Broadcast join эффективен когда:")
print("  - Одна таблица намного меньше другой (обычно < 10MB)")
print("  - Нужно избежать дорогостоящей shuffle операции")
print("  - Join с lookup/справочными таблицами")

In [ ]:
# Пример использования broadcast переменной напрямую (не для joins)
# Полезно для передачи конфигурации, констант или небольших справочников

# Создаем broadcast переменную
category_mapping = {1: 'Electronics', 2: 'Clothing', 3: 'Food', 4: 'Books', 5: 'Toys'}
broadcast_mapping = spark.sparkContext.broadcast(category_mapping)

# Используем в UDF
from pyspark.sql.functions import udf

@udf(returnType=StringType())
def get_category_name(category_id):
    # Доступ к broadcast переменной через .value
    return broadcast_mapping.value.get(category_id, 'Unknown')

# Применяем
df_with_category = df_transactions.withColumn("category_name", get_category_name(col("category_id")))
print("Использование broadcast переменной в UDF:")
df_with_category.show(10)

# Освобождаем ресурсы
broadcast_mapping.unpersist()